In [2]:
from databricks.connect import DatabricksSession
import os
from dotenv import load_dotenv
import gc
import marimo as mo
from datetime import datetime, timedelta
import pyspark.sql.functions as F
from pyspark.sql.functions import col
import polars as pl

load_dotenv()
gc.collect()

10

In [3]:
spark = DatabricksSession.builder.remote(
            host=os.getenv("DATABRICKS_HOST"),
            token=os.getenv("DATABRICKS_TOKEN")
        ).serverless(True).getOrCreate()

# Configs
Block_Table_UC = "workspace.btc_data.blocks"
block_source_External_UC = "workspace.btc_data.blocks_source"
tx_source_External_UC = "workspace.btc_data.tx_source"
# paths
tx = 's3://aws-public-blockchain/v1.0/btc/transactions/'
#dates


In [5]:
# blocks horizon
spark.sql("""

select min(date),max(date) from workspace.btc_data.blocks

""").show()

+----------+----------+
| min(date)| max(date)|
+----------+----------+
|2009-01-03|2026-09-26|
+----------+----------+



In [6]:
# tx horizon
spark.sql("""

select min(date),max(date) from workspace.btc_data.transactions

""").show()

+----------+----------+
| min(date)| max(date)|
+----------+----------+
|2026-08-08|2026-09-26|
+----------+----------+



In [9]:
spark.sql("""
-- expect 0 rows if clean
SELECT number, COUNT(*) AS c
FROM workspace.btc_data.blocks
GROUP BY number
HAVING COUNT(*) > 1
LIMIT 20;


""").show()

spark.sql("""

-- quick totals: total vs distinct should match
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT number) AS distinct_blocks
FROM workspace.btc_data.blocks;
""").show()

+------+---+
|number|  c|
+------+---+
+------+---+

+----------+---------------+
|total_rows|distinct_blocks|
+----------+---------------+
|    968501|         968501|
+----------+---------------+



In [10]:
spark.sql("""
SELECT txid, COUNT(*) AS c
FROM workspace.btc_data.transactions
GROUP BY txid
HAVING COUNT(*) > 1
LIMIT 20
""").show()

spark.sql("""
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT txid) AS distinct_tx
FROM workspace.btc_data.transactions
""").show()

+----+---+
|txid|  c|
+----+---+
+----+---+

+----------+-----------+
|total_rows|distinct_tx|
+----------+-----------+
|  31292536|   31292536|
+----------+-----------+

